In [ ]:
!pip install -q transformers datasets accelerate sentencepiece sentence-transformers evaluate peft

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    get_cosine_schedule_with_warmup,
)
from peft import get_peft_model, LoraConfig, TaskType
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine
import warnings
warnings.filterwarnings("ignore")

# ── Reproducibility ──
SEED = 42
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# ── Device ──
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_gpus = torch.cuda.device_count()
print(f"Device: {device} | GPUs available: {n_gpus}")
if torch.cuda.is_available():
    for i in range(n_gpus):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

## 1. Configuration

In [ ]:
# ══════════════════════════════════════════════════════════════
# CONFIGURATION — tweak these knobs
# ══════════════════════════════════════════════════════════════

MODEL_NAME = "google/flan-t5-large"        
EMBED_MODEL = "all-MiniLM-L6-v2"          # for cosine-similarity evaluation

# Paths (adjust for Kaggle: /kaggle/input/your-dataset-name/)
SILVER_PATH = "/kaggle/input/datasets/ahmedamrabolfadl/scientific-topic-labeling-dataset/taxonomy_pairs.csv"
GOLD_PATH   = "/kaggle/input/datasets/ahmedamrabolfadl/scientific-topic-labeling-dataset/verified_pairs.csv"

# Tokenization lengths
MAX_INPUT_LEN  = 128
MAX_TARGET_LEN = 32

# LoRA Configuration
LORA_R = 8                          # rank of LoRA adapters
LORA_ALPHA = 16                     # scaling parameter
LORA_DROPOUT = 0.05                 # dropout in LoRA layers
LORA_TARGET_MODULES = ["q", "v"]    # target linear layers in attention

# Phase 1: Silver data training
P1_EPOCHS     = 5
P1_BATCH_SIZE = 16    # per device
P1_LR         = 3e-4
P1_WARMUP     = 0.06  # fraction of total steps
P1_WEIGHT_DECAY = 0.01

# Phase 2: Gold data fine-tuning
P2_EPOCHS     = 15
P2_BATCH_SIZE = 16
P2_LR         = 1e-4  # lower LR for refinement
P2_WARMUP     = 0.10
P2_WEIGHT_DECAY = 0.01

# Early stopping patience (number of eval steps without improvement)
PATIENCE = 3

# Output directories
P1_OUTPUT_DIR = "./phase1_silver"
P2_OUTPUT_DIR = "./phase2_gold"
FINAL_MODEL_DIR = "./final_model"

## 2. Load & Prepare Data

The prompt template encodes both the research field and keyword list into a natural instruction for the model.

In [ ]:
# ── Load CSVs ──
silver_df = pd.read_csv(SILVER_PATH)
gold_df   = pd.read_csv(GOLD_PATH)

# Standardize column order
for df in [silver_df, gold_df]:
    assert set(df.columns) >= {"research_field", "topic_words", "topic_label"}, \
        f"Missing columns! Found: {df.columns.tolist()}"

silver_df = silver_df[["research_field", "topic_words", "topic_label"]].dropna().reset_index(drop=True)
gold_df   = gold_df[["research_field", "topic_words", "topic_label"]].dropna().reset_index(drop=True)

print(f"Silver dataset: {len(silver_df):,} rows")
print(f"Gold dataset:   {len(gold_df):,} rows")
silver_df.head(3)

In [ ]:
# ── Prompt template ──
PROMPT_TEMPLATE = (
    "Generate a concise topic label for the following.\n"
    "Research field: {field}\n"
    "Keywords: {words}\n"
    "Topic label:"
)

def build_prompt(row):
    return PROMPT_TEMPLATE.format(field=row["research_field"], words=row["topic_words"])

# Create input-target pairs
silver_df["input_text"]  = silver_df.apply(build_prompt, axis=1)
silver_df["target_text"] = silver_df["topic_label"]

gold_df["input_text"]  = gold_df.apply(build_prompt, axis=1)
gold_df["target_text"] = gold_df["topic_label"]

# ── Split gold data: 80% train, 20% validation ──
gold_train_df, gold_val_df = train_test_split(
    gold_df, test_size=0.20, random_state=SEED, shuffle=True
)
gold_train_df = gold_train_df.reset_index(drop=True)
gold_val_df   = gold_val_df.reset_index(drop=True)

# ── Split silver data: use 5% as held-out for Phase 1 validation ──
silver_train_df, silver_val_df = train_test_split(
    silver_df, test_size=0.05, random_state=SEED, shuffle=True
)
silver_train_df = silver_train_df.reset_index(drop=True)
silver_val_df   = silver_val_df.reset_index(drop=True)

print(f"Silver  train: {len(silver_train_df):,} | Silver  val: {len(silver_val_df):,}")
print(f"Gold    train: {len(gold_train_df):,} | Gold    val: {len(gold_val_df):,}")
print(f"\nSample prompt:\n{silver_df['input_text'].iloc[0]}")
print(f"Target: {silver_df['target_text'].iloc[0]}")

## 3. Tokenizer, Model & Dataset

In [ ]:
# ── Tokenizer & Model ──
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# ── Apply LoRA ──
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
    target_modules=LORA_TARGET_MODULES,
)
model = get_peft_model(model, lora_config)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable:        {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
model.print_trainable_parameters()

In [ ]:
from datasets import Dataset as HFDataset

def tokenize_fn(examples):
    """Tokenize inputs and targets for Seq2SeqTrainer."""
    model_inputs = tokenizer(
        examples["input_text"],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding=False,                    # dynamic padding via data collator
    )
    labels = tokenizer(
        text_target=examples["target_text"],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding=False,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

def make_hf_dataset(df):
    """Convert a DataFrame with input_text/target_text to a tokenized HF Dataset."""
    ds = HFDataset.from_pandas(df[["input_text", "target_text"]])
    ds = ds.map(tokenize_fn, batched=True, remove_columns=["input_text", "target_text"])
    return ds

# Build all tokenized datasets
silver_train_ds = make_hf_dataset(silver_train_df)
silver_val_ds   = make_hf_dataset(silver_val_df)
gold_train_ds   = make_hf_dataset(gold_train_df)
gold_val_ds     = make_hf_dataset(gold_val_df)

print(f"Silver train tokens: {silver_train_ds}")
print(f"Gold val tokens:     {gold_val_ds}")

## 4. Evaluation Metric — Cosine Similarity

We load a sentence-transformer once and use it throughout training and final evaluation.

In [ ]:
# ── Sentence embedding model (for evaluation only) ──
embed_model = SentenceTransformer(EMBED_MODEL, device="cpu")  # keep off GPU to save VRAM

def mean_cosine_similarity(preds: list[str], targets: list[str]) -> float:
    """Compute mean pairwise cosine similarity between two lists of strings."""
    pred_embs   = embed_model.encode(preds,   batch_size=128, show_progress_bar=False, convert_to_numpy=True)
    target_embs = embed_model.encode(targets, batch_size=128, show_progress_bar=False, convert_to_numpy=True)
    # Row-wise cosine similarity
    sims = np.array([
        sk_cosine(p.reshape(1, -1), t.reshape(1, -1))[0, 0]
        for p, t in zip(pred_embs, target_embs)
    ])
    return float(sims.mean())

def compute_metrics(eval_preds):
    """
    Called by Seq2SeqTrainer during evaluation.
    eval_preds = (predictions_ids, label_ids)
    """
    preds_ids, labels_ids = eval_preds

    # Replace -100 (ignore index) with pad token for decoding
    labels_ids = np.where(labels_ids != -100, labels_ids, tokenizer.pad_token_id)

    decoded_preds  = tokenizer.batch_decode(preds_ids,  skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels_ids, skip_special_tokens=True)

    # Strip whitespace
    decoded_preds  = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    cos_sim = mean_cosine_similarity(decoded_preds, decoded_labels)
    return {"cosine_similarity": cos_sim}

# Quick sanity check
test_sim = mean_cosine_similarity(
    ["Natural Language Processing", "Computer Vision"],
    ["NLP and Text Mining", "Image Recognition"]
)
print(f"Sanity check cosine sim: {test_sim:.4f}")

## 5. Phase 1 — Silver Data Pre-training

Train on the large silver dataset. We validate against both a held-out silver split and the gold validation set. Early stopping is based on cosine similarity on the silver validation set.

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True, pad_to_multiple_of=8)

p1_steps_per_epoch = len(silver_train_ds) // (P1_BATCH_SIZE * max(n_gpus, 1))
p1_total_steps = p1_steps_per_epoch * P1_EPOCHS
p1_eval_steps = max(p1_steps_per_epoch // 2, 1)  # evaluate twice per epoch

print(f"Phase 1: {p1_total_steps} total steps, eval every {p1_eval_steps} steps")

p1_args = Seq2SeqTrainingArguments(
    output_dir=P1_OUTPUT_DIR,
    num_train_epochs=P1_EPOCHS,
    per_device_train_batch_size=P1_BATCH_SIZE,
    per_device_eval_batch_size=P1_BATCH_SIZE * 2,
    learning_rate=P1_LR,
    weight_decay=P1_WEIGHT_DECAY,
    warmup_steps=int(p1_total_steps * P1_WARMUP),
    lr_scheduler_type="cosine",
    fp16=torch.cuda.is_available(),
    eval_strategy="steps",
    eval_steps=p1_eval_steps,
    save_strategy="steps",
    save_steps=p1_eval_steps,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="cosine_similarity",
    greater_is_better=True,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    logging_steps=50,
    report_to="none",
    dataloader_num_workers=2,
    seed=SEED,
    label_smoothing_factor=0.1,            # mild label smoothing for regularization
    gradient_accumulation_steps=1,
)

p1_trainer = Seq2SeqTrainer(
    model=model,
    args=p1_args,
    train_dataset=silver_train_ds,
    eval_dataset=silver_val_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
)

print("Starting Phase 1 training (silver data)...")
p1_trainer.train()
print("Phase 1 complete!")

In [ ]:
# ── Evaluate Phase 1 on gold validation set ──
print("Phase 1 → evaluating on gold validation set:")
p1_gold_results = p1_trainer.evaluate(eval_dataset=gold_val_ds)
print(f"  Gold val cosine similarity: {p1_gold_results['eval_cosine_similarity']:.4f}")
print(f"  Gold val loss:              {p1_gold_results['eval_loss']:.4f}")

## 6. Phase 2 — Gold Data Fine-tuning

Fine-tune the Phase-1 model on the smaller, high-quality gold dataset. This refines the model to produce semantically accurate labels while preserving the formatting learned from the silver data. Lower learning rate to avoid catastrophic forgetting.

In [ ]:
p2_steps_per_epoch = len(gold_train_ds) // (P2_BATCH_SIZE * max(n_gpus, 1))
p2_total_steps = p2_steps_per_epoch * P2_EPOCHS
p2_eval_steps = max(p2_steps_per_epoch, 1)  # evaluate every epoch

print(f"Phase 2: {p2_total_steps} total steps, eval every {p2_eval_steps} steps")

p2_args = Seq2SeqTrainingArguments(
    output_dir=P2_OUTPUT_DIR,
    num_train_epochs=P2_EPOCHS,
    per_device_train_batch_size=P2_BATCH_SIZE,
    per_device_eval_batch_size=P2_BATCH_SIZE * 2,
    learning_rate=P2_LR,
    weight_decay=P2_WEIGHT_DECAY,
    warmup_steps=int(p2_total_steps * P2_WARMUP),
    lr_scheduler_type="cosine",
    fp16=torch.cuda.is_available(),
    eval_strategy="steps",
    eval_steps=p2_eval_steps,
    save_strategy="steps",
    save_steps=p2_eval_steps,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="cosine_similarity",
    greater_is_better=True,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    logging_steps=10,
    report_to="none",
    dataloader_num_workers=2,
    seed=SEED,
    label_smoothing_factor=0.05,            # lighter smoothing for gold data
    gradient_accumulation_steps=1,
)

# The model object is already the best Phase 1 model (load_best_model_at_end)
p2_trainer = Seq2SeqTrainer(
    model=model,
    args=p2_args,
    train_dataset=gold_train_ds,
    eval_dataset=gold_val_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
)

print("Starting Phase 2 training (gold data)...")
p2_trainer.train()
print("Phase 2 complete!")

## 7. Save Final Model

In [ ]:
# ── Save best model & LoRA adapter ──
model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print(f"LoRA adapter and tokenizer saved to {FINAL_MODEL_DIR}/")

## 8. Full Evaluation on Gold Validation Set

Generate labels for every gold validation example, compute per-sample and aggregate cosine similarity, and display results.

In [ ]:
@torch.no_grad()
def generate_labels(input_texts: list[str], batch_size: int = 32) -> list[str]:
    """Generate topic labels from input prompts using the fine-tuned model."""
    model.eval()
    all_preds = []
    for i in range(0, len(input_texts), batch_size):
        batch = input_texts[i : i + batch_size]
        inputs = tokenizer(
            batch,
            max_length=MAX_INPUT_LEN,
            truncation=True,
            padding=True,
            return_tensors="pt",
        ).to(model.device)
        outputs = model.generate(
            **inputs,
            max_length=MAX_TARGET_LEN,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3,
        )
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        all_preds.extend([d.strip() for d in decoded])
    return all_preds

# ── Generate predictions on gold validation set ──
val_inputs  = gold_val_df["input_text"].tolist()
val_targets = gold_val_df["target_text"].tolist()

val_preds = generate_labels(val_inputs)

# ── Per-sample cosine similarity ──
pred_embs   = embed_model.encode(val_preds,   batch_size=128, convert_to_numpy=True)
target_embs = embed_model.encode(val_targets, batch_size=128, convert_to_numpy=True)

per_sample_sim = np.array([
    sk_cosine(p.reshape(1, -1), t.reshape(1, -1))[0, 0]
    for p, t in zip(pred_embs, target_embs)
])

print(f"{'='*60}")
print(f"FINAL RESULTS — Gold Validation Set ({len(val_preds)} samples)")
print(f"{'='*60}")
print(f"Mean Cosine Similarity:   {per_sample_sim.mean():.4f}")
print(f"Median Cosine Similarity: {np.median(per_sample_sim):.4f}")
print(f"Std Dev:                  {per_sample_sim.std():.4f}")
print(f"Min:                      {per_sample_sim.min():.4f}")
print(f"Max:                      {per_sample_sim.max():.4f}")
print(f"% samples ≥ 0.8:         {(per_sample_sim >= 0.8).mean() * 100:.1f}%")
print(f"% samples ≥ 0.9:         {(per_sample_sim >= 0.9).mean() * 100:.1f}%")

In [ ]:
# ── Show sample predictions ──
results_df = pd.DataFrame({
    "research_field": gold_val_df["research_field"].values,
    "keywords": gold_val_df["topic_words"].values,
    "target_label": val_targets,
    "predicted_label": val_preds,
    "cosine_sim": per_sample_sim.round(4),
})

# Show best and worst predictions
print("\n🔝 TOP 10 (highest similarity):\n")
print(results_df.nlargest(10, "cosine_sim")[["target_label", "predicted_label", "cosine_sim"]].to_string(index=False))

print(f"\n\n🔻 BOTTOM 10 (lowest similarity):\n")
print(results_df.nsmallest(10, "cosine_sim")[["target_label", "predicted_label", "cosine_sim"]].to_string(index=False))

In [ ]:
# ── Distribution plot ──
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histogram
axes[0].hist(per_sample_sim, bins=30, edgecolor="black", alpha=0.75, color="steelblue")
axes[0].axvline(per_sample_sim.mean(), color="red", linestyle="--", label=f"Mean: {per_sample_sim.mean():.3f}")
axes[0].set_xlabel("Cosine Similarity")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of Cosine Similarities")
axes[0].legend()

# Sorted plot
axes[1].plot(np.sort(per_sample_sim), color="steelblue")
axes[1].axhline(0.8, color="orange", linestyle="--", alpha=0.7, label="0.8 threshold")
axes[1].axhline(0.9, color="green", linestyle="--", alpha=0.7, label="0.9 threshold")
axes[1].set_xlabel("Sample (sorted)")
axes[1].set_ylabel("Cosine Similarity")
axes[1].set_title("Sorted Cosine Similarities")
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. Inference Example

Load the saved model and generate a label for a new input.

In [ ]:
# ── Reload model with LoRA for inference demo ──
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

inf_base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
inf_model = PeftModel.from_pretrained(inf_base_model, FINAL_MODEL_DIR)
inf_tokenizer = AutoTokenizer.from_pretrained(FINAL_MODEL_DIR)
inf_model.eval()

def predict_topic_label(research_field: str, keywords: str) -> str:
    """Generate a topic label given a research field and comma-separated keywords."""
    prompt = PROMPT_TEMPLATE.format(field=research_field, words=keywords)
    inputs = inf_tokenizer(prompt, max_length=MAX_INPUT_LEN, truncation=True, return_tensors="pt").to(device)
    with torch.no_grad():
        output = inf_model.generate(
            **inputs,
            max_length=MAX_TARGET_LEN,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3,
        )
    return inf_tokenizer.decode(output[0], skip_special_tokens=True).strip()

# ── Example predictions ──
examples = [
    ("Machine Learning", "neural, network, deep, learning, convolutional"),
    ("Economics", "inflation, monetary, policy, central, bank"),
    ("Biology", "gene, expression, regulation, transcription, protein"),
    ("Computer Science", "graph, algorithm, optimization, complexity, polynomial"),
]

print(f"{'Research Field':<25} {'Keywords':<55} {'Predicted Label'}")
print("─" * 120)
for field, keywords in examples:
    label = predict_topic_label(field, keywords)
    print(f"{field:<25} {keywords:<55} {label}")

## 10. Export Results

In [ ]:
# ── Save evaluation results ──
results_df.to_csv("evaluation_results.csv", index=False)
print(f"Evaluation results saved to evaluation_results.csv ({len(results_df)} rows)")

# ── Training summary ──
print(f"\n{'='*60}")
print("TRAINING SUMMARY")
print(f"{'='*60}")
print(f"Model:               {MODEL_NAME}")
print(f"Silver data:         {len(silver_df):,} rows ({P1_EPOCHS} epochs)")
print(f"Gold data:           {len(gold_df):,} rows ({P2_EPOCHS} epochs)")
print(f"Phase 1 best metric: {p1_trainer.state.best_metric:.4f} (silver val cos-sim)")
print(f"Phase 2 best metric: {p2_trainer.state.best_metric:.4f} (gold val cos-sim)")
print(f"Final mean cos-sim:  {per_sample_sim.mean():.4f} (gold val)")
print(f"Model saved to:      {FINAL_MODEL_DIR}/")